# 🇸🇩 تدريب وضبط المحرك الصوتي العصبي للهجة السودانية الأصيلة
### Sudanese Dialect Neural Fine-Tuning (Coqui XTTS-v2 GPT Fine-Tuning)
---
هذا الدفتر مخصص لتدريب النموذج العصبي ليتحول من نطق الفصحى الجامد إلى **النطق السوداني البشري الأصيل**.
يقوم بالتدريب على آلاف العينات الصوتية المفرغة من الإذاعة والتلفزيون والبودكاست والدراما السودانية.

**خطوات التدريب:**
1. `▶` **الخلية 1**: تجهيز بيئة التدريب ومكتبات XTTS Trainer.
2. `▶` **الخلية 2**: ربط مجموعة البيانات المفرغة (`metadata.csv` + العينات الصوتية في Drive).
3. `▶` **الخلية 3**: بدء عملية التدريب العصبي (Fine-Tuning) وحفظ النموذج المطوّر في Drive.
4. `▶` **الخلية 4**: تجربة وسماع الصوت بعد التدريب بالمخارج السودانية الحقيقية!

## 1️⃣ الخلية الأولى: تثبيت مكتبات ومحرك التدريب الصوتي
*(تستغرق حوالي دقيقة إلى دقيقتين)*

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# تثبيت الحزم المطلوبة لتدريب XTTS-v2
!pip install -q coqui-tts soundfile gradio
!pip install -q git+https://github.com/coqui-ai/trainer

# ربط Google Drive
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('\n✅ تم تجهيز البيئة بنجاح وجاهزة لبدء التدريب!')

## 2️⃣ الخلية الثانية: فحص وتجهيز مجموعة بيانات اللهجة السودانية (Dataset Audit)
*(تقرأ ملف `metadata.csv` والعينات المجهزة من الـ 218 ملفاً)*

In [ ]:
import pandas as pd
import os

DATASET_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuning_dataset'
METADATA_FILE = os.path.join(DATASET_DIR, 'metadata.csv')
WAVS_DIR = os.path.join(DATASET_DIR, 'wavs')

if os.path.exists(METADATA_FILE):
    df = pd.read_csv(METADATA_FILE, sep='|', on_bad_lines='skip')
    print(f'📊 إجمالي عينات التدريب الجاهزة: {len(df):,} عينة صوتية مقطوعة!')
    print('\n--- عينة من بيانات التدريب السودانية ---')
    display(df.head(5))
else:
    print('⚠️ ملف metadata.csv قيد التوليد حالياً في الخلفية. انتظر لحظات!')


## 3️⃣ الخلية الثالثة: إطلاق واجهة التدريب التفاعلية (XTTS WebUI Trainer)
*(تفتح لك واجهة كاملة بنقرة واحدة لتدريب النموذج ومراقبة التطور خطوة بخطوة)*

In [ ]:
# استنساخ وتشغيل واجهة التدريب الرسمية المطورة لـ XTTS
!git clone -q https://github.com/daswer123/xtts-finetune-webui
%cd xtts-finetune-webui
!pip install -q -r requirements.txt

# نسخ بيانات التدريب السودانية إلى مجلد العمل للتدريب الفوري
!python app.py --share


## 4️⃣ الخلية الرابعة: تجربة وسماع الصوت بعد التدريب العصبي
*(اختبار فوري لنطق الكلمات السودانية الصعبة التي كان يتلعثم فيها)*

In [ ]:
import torch
from TTS.api import TTS
from IPython.display import Audio, display
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# فحص ما إذا كان النموذج المدرب متاحاً
FINE_TUNED_MODEL_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/fine_tuned_model'
REF_WAV = '/content/drive/MyDrive/Sudanese_Voice_Project/corpus/references/clean_podcast_8s.wav'

if os.path.exists(os.path.join(FINE_TUNED_MODEL_DIR, 'model.pth')):
    print('🎉 تم العثور على النموذج السوداني المدرب!')
    xtts_tuned = TTS(model_path=FINE_TUNED_MODEL_DIR, config_path=os.path.join(FINE_TUNED_MODEL_DIR, 'config.json')).to(device)
else:
    print('ℹ️ جاري استخدام النموذج الأساسي مع المعايرة حتى يكتمل حفظ التدريب.')
    xtts_tuned = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

test_phrase = 'يا زول مرحب بيك في استوديو الصوت السوداني، دحين كدا النطق بقى سمح بالحيل وظابط تمام.'
out_audio = '/content/drive/MyDrive/Sudanese_Voice_Project/cloned_sudanese.wav'

xtts_tuned.tts_to_file(
    text=test_phrase,
    speaker_wav=REF_WAV,
    language='ar',
    file_path=out_audio,
    temperature=0.6,
    repetition_penalty=2.0
)

print('✅ تم التوليد بنجاح!')
display(Audio(out_audio, autoplay=True))
